# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hammad-vibes/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (Refresh / Content Opportunity Scoring)

### My lane as an ML task

**Task type:** Scoring (a ranking-oriented ML task).

**Problem:** Identify which content pages are the strongest candidates for a refresh, so the content team can prioritize its work.

**What the model should do:** Assign each content page an opportunity score. A higher score means the page should receive more attention for a possible refresh.

**Action supported:** Help the content team decide which pages to review and update first.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

The goal is to prioritize content pages for a possible refresh.

The starter dataset does not contain a directly observed outcome showing whether a refresh actually improved a page. Therefore, I will use `is_declining_label` as a proxy for identifying pages with declining search impressions.

This label is 1 when `trend_direction` is `down`, and 0 otherwise. However, it is derived from `trend_pct`, so neither `trend_direction` nor `trend_pct` will be used as model features.

This proxy identifies historical decline, not whether refreshing a page will improve its performance. The output will support editorial decisions rather than automatically determine which pages must be updated.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

The primary metric will be Precision@K, where K is the number of highest-priority pages selected for review.

Precision@K measures the proportion of selected pages that have the declining label. For example, Precision@20 tells us how many of the top 20 selected pages are actually labelled as declining.

This metric matches the goal of helping an editorial team prioritize a limited number of pages for review. Since the label is only a proxy, a high score does not prove that refreshing these pages will improve performance.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!git clone https://github.com/Hammad-vibes/FlyRank-Internship.git
%cd FlyRank-Internship

import pandas as pd
from pathlib import Path

csv_path = Path("data/raw/content_refresh_anonymized.csv")

if not csv_path.exists():
    raise FileNotFoundError(
        "CSV not found. Check the repository path in your notebook environment."
    )

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Unique content pages:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())


display(df[[
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "trend_pct",
    "trend_direction"
]].head(10))




Cloning into 'FlyRank-Internship'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 127 (delta 39), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.87 MiB | 6.91 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/FlyRank-Internship/FlyRank-Internship/FlyRank-Internship
Dataset shape: (30000, 44)
Unique content pages: 30000
Unique clients: 32


,content_id,client_id,content_type,impressions_90d,clicks_90d,trend_pct,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,-41.4,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,-57.7,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,-60.9,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,-13.8,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,-34.7,down
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,-38.9,down
6,content_9a34b442b552,client_8722616204,keyword article,20,0,-92.3,down
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.6,stable
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,-58.8,down
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,-29.2,down


In [9]:

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Target label distribution:")
print(df["is_declining_label"].value_counts())

display(df[[
    "content_id",
    "impressions_90d",
    "trend_direction",
    "is_declining_label"
]].head(10))


Target label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


,content_id,impressions_90d,trend_direction,is_declining_label
0,content_304f48230142,3803,down,1
1,content_a1fb4e703a9e,15320,down,1
2,content_9aa793d4d895,12581,down,1
3,content_331d6c4de07b,11751,stable,0
4,content_d99b7a2d90ca,19140,down,1
5,content_d4084a4bc775,3970,down,1
6,content_9a34b442b552,20,down,1
7,content_a63219c6e95a,1724,stable,0
8,content_5e6c160719bc,32574,down,1
9,content_c27558df2b0c,1240,down,1


## 5. Why ML beats a fixed rule here

A fixed rule could prioritize pages based only on declining impressions. However, content performance is influenced by multiple factors, including impressions, clicks, pageviews, content age, update recency, and content type.

An ML model could combine these signals to help prioritize pages more consistently than a single manually chosen threshold. Its usefulness must be tested against a simple rule-based baseline rather than assumed.

The current starter dataset contains aggregated trailing-90-day metrics and a historical decline proxy. It does not show whether a refresh actually improved a page. Therefore, this task is decision support for editorial review, not a proven method for predicting refresh success.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.